# ⚗️ IRMS Analysis: Geographic Origin of Olive Oil
### FBMFOR — Food Fraud Analysis | University of Reading

This notebook loads the stable-isotope ratio data from `data/irms/irms.csv`,
pivots it from long to wide format, plots the δ²H / δ¹⁸O geographic-origin map
and a δ¹³C C3/C4 adulteration strip, and assigns each sample to the nearest
published reference region.


## 1 · Setup

In [ ]:
# ============================================================
# SETUP — INSTALL AND IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import matplotlib.patheffects as pe
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab      # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Paths ─────────────────────────────────────────────────────
if IN_COLAB:
    DATA_FILE  = Path('/content/data/irms/irms.csv')
    OUTPUT_DIR = Path('/content/data/processed')
    print("\u25b6 Running on Google Colab")
else:
    DATA_FILE  = Path('../data/irms/irms.csv')
    OUTPUT_DIR = Path('../data/processed')
    print("\u25b6 Running locally")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Download data from GitHub when running in Colab ───────────
GITHUB_RAW = "https://raw.githubusercontent.com/ggkuhnle/fbmfor-foodfraud/main"
if IN_COLAB and not DATA_FILE.exists():
    import urllib.request
    DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
    url = f"{GITHUB_RAW}/data/irms/irms.csv"
    print(f"  Downloading {url} ...")
    urllib.request.urlretrieve(url, DATA_FILE)
    print("  Done \u2713")

print(f"   Data file      : {DATA_FILE.resolve()}")
print(f"   Output dir     : {OUTPUT_DIR.resolve()}")
print("\nLibraries loaded \u2713")


## 2 · Load & reshape data

The instrument export uses a **long format**: one row per (sample, isotope) pair.

| Column | Content |
|--------|---------|
| `Sample` | Sample identifier |
| `delta` | Isotope ratio value (‰) |
| `IR` | Isotope system: `d13C`, `d2H`, or `d18O` |

We pivot to wide format so each row is one sample and each column is one isotope.
Sample-name variants (`Oil1` / `Oil 1`) are normalised automatically.


In [ ]:
# ============================================================
# LOAD AND RESHAPE IRMS DATA
# ============================================================

# ── Read raw CSV (trailing comma produces an unnamed extra column) ────────────
raw = pd.read_csv(DATA_FILE)
raw.columns = [c.strip() for c in raw.columns]

# Drop any fully-NaN columns (artifact of trailing commas in export)
raw = raw.dropna(axis=1, how='all')
raw = raw.dropna(subset=['Sample', 'delta', 'IR'])

print("Raw data:")
print(raw.to_string(index=False))
print()

# ── Normalise sample names ────────────────────────────────────────────────────
# Remove interior spaces so "Oil 1" and "Oil1" become the same key.
raw['Sample'] = raw['Sample'].str.strip().str.replace(r'\s+', '', regex=True)

# ── Normalise IR column ───────────────────────────────────────────────────────
raw['IR'] = raw['IR'].str.strip()

# ── Pivot long → wide ────────────────────────────────────────────────────────
df = (raw.pivot_table(index='Sample', columns='IR', values='delta', aggfunc='mean')
        .reset_index()
        .rename_axis(None, axis=1))

# Rename Sample → SampleID for downstream compatibility
df = df.rename(columns={'Sample': 'SampleID'})
df.columns.name = None

print("Wide-format table:")
print(df.to_string(index=False))
print()

# ── Detect available isotope columns ─────────────────────────────────────────
HAS_13C  = 'd13C'  in df.columns
HAS_2H   = 'd2H'   in df.columns
HAS_18O  = 'd18O'  in df.columns
HAS_HO   = HAS_2H and HAS_18O

iso_present = [c for c in ['d13C', 'd2H', 'd18O', 'd15N'] if c in df.columns]
print(f"Isotopes detected : {iso_present}")
print(f"H/O origin plot   : {'yes' if HAS_HO else 'no  (need d2H + d18O)'}")
print(f"C3/C4 strip       : {'yes' if HAS_13C else 'no  (need d13C)'}")


## 3 · Literature reference regions

Torres-Cobos and colleages have analysed olive oils from a range of different regions to identify Italian and non-Italian sources. The median (IQR) values are shown in the table below.

| Region   | δ${18}$O (‰)       | δ${2}$H (‰)              | δ${13}$C (‰)          |
|----------|--------------------|--------------------------|-----------------------|
| Italy    | 24.0 (23.3 -25.1)  | −144.0 (-148.8 – -140.0) | -29.7 (-30.1 – -29.1) |
| Apulia   | 23.7 (23.0 – 24.8) | -142.0 (-146.4 – -139.0) | -29.9 (-30.2 – -29.5) |
| Calabria | 24.5 (23.5 – 25.9) | -141.8 (-145.0 – -135.8) | -29.5 (-29.9 – -29.0) |
| Sicily   | 25.1 (24.4 – 25.7) | -146.5 (-150.8 – -143.7) | -28.9 (-29.1 – -28.9) |
| Spain    | 26.3 (24.2 – 27.0) | −145.8 (-148.4 – -140.8) | -29.7 (-29.9 – -29.3) |
| Greece   | 25.1 (24.6 – 25.5) | −147.2 (-148.4 – -140.8) | -29.2 (-29.7 – -28.8) |
| Portugal | 29.5 (28.9 – 30.5) | −132.8 (-148.4 – -140.8) | -29.9 (-30.0 – -29.2) |
| Tunisia  | 25.6 (25.3 – 26.4) | −142.5 (-143.3 – -141.4) | -30.1 (-30.0 – -29.2) |
| Türkiye  | 26.0 (25.3 – 26.7) | −137.6 (-144.6 – -131.2) | -29.2 (-29.7 – -28.5) |


> Torres-Cobos et al. (2025) *Food Chem.* doi.org/10.1016/j.foodchem.2025.143655



In [ ]:
# ============================================================
# LITERATURE REFERENCE REGIONS
# ============================================================

REF_HO = {
    "Italy": {
        "d18O_mean": 24.0, "d18O_sd": 0.46,
        "d2H_mean": -144.0, "d2H_sd": 2.24,
        "d13C_mean": -29.7, "d13C_sd": 0.26,
        "colour": "#2E86AB", "n": 242
    },
    "Apulia": {
        "d18O_mean": 23.7, "d18O_sd": 0.46,
        "d2H_mean": -142.0, "d2H_sd": 1.89,
        "d13C_mean": -29.9, "d13C_sd": 0.18,
        "colour": "#5DA5DA", "n": 73
    },
    "Calabria": {
        "d18O_mean": 24.5, "d18O_sd": 0.61,
        "d2H_mean": -141.8, "d2H_sd": 2.35,
        "d13C_mean": -29.5, "d13C_sd": 0.23,
        "colour": "#60BD68", "n": 58
    },
    "Sicily": {
        "d18O_mean": 25.1, "d18O_sd": 0.33,
        "d2H_mean": -146.5, "d2H_sd": 1.81,
        "d13C_mean": -28.9, "d13C_sd": 0.05,
        "colour": "#F17CB0", "n": 40
    },
    "Spain": {
        "d18O_mean": 26.3, "d18O_sd": 0.71,
        "d2H_mean": -145.8, "d2H_sd": 1.94,
        "d13C_mean": -29.7, "d13C_sd": 0.15,
        "colour": "#E84855", "n": 51
    },
    "Greece": {
        "d18O_mean": 25.1, "d18O_sd": 0.23,
        "d2H_mean": -147.2, "d2H_sd": 1.94,
        "d13C_mean": -29.2, "d13C_sd": 0.23,
        "colour": "#3BB273", "n": 39
    },
    "Portugal": {
        "d18O_mean": 29.5, "d18O_sd": 0.41,
        "d2H_mean": -132.8, "d2H_sd": 2.04,
        "d13C_mean": -29.9, "d13C_sd": 0.20,
        "colour": "#B07AA1", "n": 23
    },
    "Tunisia": {
        "d18O_mean": 25.6, "d18O_sd": 0.28,
        "d2H_mean": -142.5, "d2H_sd": 0.48,
        "d13C_mean": -30.1, "d13C_sd": 0.20,
        "colour": "#F4A261", "n": 17
    },
    "Türkiye": {
        "d18O_mean": 26.0, "d18O_sd": 0.36,
        "d2H_mean": -137.6, "d2H_sd": 3.42,
        "d13C_mean": -29.2, "d13C_sd": 0.31,
        "colour": "#4E79A7", "n": 21
    },
}

print("✓ Reference data loaded")
for r, v in REF_HO.items():
    print(f"  {r:<10}  δ¹⁸O = {v['d18O_mean']:.1f} ± {v['d18O_sd']:.1f}‰  "
          f"δ²H = {v['d2H_mean']:.1f} ± {v['d2H_sd']:.1f}‰  (n={v['n']})")


## 4 · Summary statistics


In [ ]:
# ============================================================
# SUMMARY STATISTICS
# ============================================================

col_labels = {
    "d13C": "δ¹³C (‰ VPDB)",
    "d2H":  "δ²H  (‰ VSMOW)",
    "d18O": "δ¹⁸O (‰ VSMOW)",
    "d15N": "δ¹⁵N (‰ AIR)",
}

print(f"  {'Isotope':<20}  {'Mean':>8}  {'SD':>8}  {'Min':>8}  {'Max':>8}  {'N':>4}")
print("  " + "─" * 58)
for col in iso_present:
    vals = df[col].dropna()
    if len(vals) > 0:
        mean = vals.mean() if len(vals) > 1 else vals.iloc[0]
        sd   = vals.std()  if len(vals) > 1 else float('nan')
        print(f"  {col_labels.get(col, col):<20}  "
              f"{mean:8.2f}  {sd:8.2f}  {vals.min():8.2f}  {vals.max():8.2f}  {len(vals):4d}")

print()
print(df[['SampleID'] + iso_present].to_string(index=False))


## 5 · δ²H vs δ¹⁸O — Geographic Origin Plot

Both hydrogen and oxygen isotope ratios in plant-derived fats reflect
the **isotopic composition of local meteoric water**, which varies
predictably with latitude, altitude, and distance from the coast.
Italian oils (northern Mediterranean) cluster at lower δ¹⁸O than
Spanish or Tunisian oils.


In [ ]:
# ============================================================
# δ²H vs δ¹⁸O  — GEOGRAPHIC ORIGIN PLOT
# ============================================================

if not HAS_HO:
    print("Skipped — need both d2H and d18O columns.")
else:
    ELLIPSE_SD = 1.0
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_facecolor("#FAFAFA")

    # ── Reference ellipses ────────────────────────────────────────────────────
    for region, vals in REF_HO.items():
        ell = Ellipse(
            (vals["d18O_mean"], vals["d2H_mean"]),
            width=2 * ELLIPSE_SD * vals["d18O_sd"],
            height=2 * ELLIPSE_SD * vals["d2H_sd"],
            facecolor=vals["colour"], alpha=0.18,
            edgecolor=vals["colour"], linewidth=1.5, linestyle="--", zorder=1,
        )
        ax.add_patch(ell)
        ax.text(vals["d18O_mean"], vals["d2H_mean"], region,
                fontsize=9, color=vals["colour"], fontweight="bold",
                ha="center", va="center", zorder=3,
                path_effects=[pe.withStroke(linewidth=2, foreground="white")])

    # ── Sample points ─────────────────────────────────────────────────────────
    for _, row in df.iterrows():
        ax.scatter(row["d18O"], row["d2H"],
                   s=90, color="#2C3E50", edgecolors="white",
                   linewidths=0.9, zorder=5)
        ax.annotate(str(row["SampleID"]),
                    xy=(row["d18O"], row["d2H"]),
                    xytext=(row["d18O"] + 0.15, row["d2H"] + 1.5),
                    fontsize=9, color="#2C3E50", fontweight="bold",
                    path_effects=[pe.withStroke(linewidth=2, foreground="white")],
                    zorder=6)

    # ── Decoration ───────────────────────────────────────────────────────────
    ax.set_xlabel(r"$\delta^{18}$O (‰ vs VSMOW)", fontsize=12)
    ax.set_ylabel(r"$\delta^{2}$H (‰ vs VSMOW)", fontsize=12)
    ax.set_title(r"Stable Isotope Map: $\delta^{2}$H vs $\delta^{18}$O"
                 "\nGeographic Origin of Olive Oil",
                 fontsize=13, fontweight="bold", pad=10)
    legend_handles = [
        mpatches.Patch(facecolor=v["colour"], alpha=0.5,
                       edgecolor=v["colour"], linestyle="--",
                       label=f"{r} (n={v['n']})")
        for r, v in REF_HO.items()
    ] + [plt.scatter([], [], s=70, color="#2C3E50",
                     edgecolors="white", linewidths=0.8, label="Sample")]
    ax.legend(handles=legend_handles,
              title=f"Reference regions (±{ELLIPSE_SD} SD)",
              title_fontsize=8.5, fontsize=8.5, loc="upper left", framealpha=0.9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.text(0.5, -0.10,
            "Ellipses: ±1 SD reference regions "
            "(Torres-Cobos et al. 2025; Longobardi et al. 2012).",
            transform=ax.transAxes, ha="center", fontsize=8, color="grey")
    plt.tight_layout()
    out_path = OUTPUT_DIR / 'irms_HO_origin.png'
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")


## 6 · δ¹³C — C3 / C4 Adulteration Strip

All genuine olive oil is a **C3 plant product** (δ¹³C ≈ −32 to −22‰).
Corn oil is a **C4 product** (δ¹³C ≈ −16 to −9‰).
Even a 5–10 % corn oil addition shifts the δ¹³C measurably towards less
negative values, making IRMS a sensitive fraud detector.


In [ ]:
# ============================================================
# δ13C STRIP PLOT — GEOGRAPHY + C3/C4 SCREEN
# ============================================================

if not HAS_13C:
    print("Skipped — no d13C column.")
else:
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.patheffects as pe
    from matplotlib.lines import Line2D

    # ------------------------------------------------------------------
    # Reference values from your table
    # Mean and approximate SD inferred from the reported bracketed range
    # as SD ≈ (upper - lower) / 3.92
    # ------------------------------------------------------------------
    REF_C13 = {
        "Italy":    {"mean": -29.7, "sd": 0.26, "colour": "#2E86AB", "n": 242},
        "Apulia":   {"mean": -29.9, "sd": 0.18, "colour": "#5DA5DA", "n": 73},
        "Calabria": {"mean": -29.5, "sd": 0.23, "colour": "#60BD68", "n": 58},
        "Sicily":   {"mean": -28.9, "sd": 0.05, "colour": "#F17CB0", "n": 40},
        "Spain":    {"mean": -29.7, "sd": 0.15, "colour": "#E84855", "n": 51},
        "Greece":   {"mean": -29.2, "sd": 0.23, "colour": "#3BB273", "n": 39},
        "Portugal": {"mean": -29.9, "sd": 0.20, "colour": "#B07AA1", "n": 23},
        "Tunisia":  {"mean": -30.1, "sd": 0.20, "colour": "#F4A261", "n": 17},
        "Türkiye":  {"mean": -29.2, "sd": 0.31, "colour": "#4E79A7", "n": 21},
    }

    # Broad chemistry screen
    # Adjust these if you use different reference limits
    C3_RANGE = (-32.5, -27.0)
    C4_RANGE = (-22.0, -10.0)

    # Combined olive-oil reference envelope across all regions
    olive_low = min(v["mean"] - v["sd"] for v in REF_C13.values())
    olive_high = max(v["mean"] + v["sd"] for v in REF_C13.values())

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_facecolor("#FAFAFA")

    # ------------------------------------------------------------------
    # Background zones
    # ------------------------------------------------------------------
    ax.axvspan(*C3_RANGE, alpha=0.10, color="#2E86AB", zorder=0)
    ax.axvspan(*C4_RANGE, alpha=0.10, color="#E76F51", zorder=0)
    ax.axvspan(olive_low, olive_high, alpha=0.12, color="#8BC34A", zorder=0)

    # ------------------------------------------------------------------
    # Plot regional reference bands as horizontal rows
    # ------------------------------------------------------------------
    regions = list(REF_C13.keys())
    y_positions = np.arange(len(regions), 0, -1)

    for y, region in zip(y_positions, regions):
        ref = REF_C13[region]
        mean = ref["mean"]
        sd = ref["sd"]
        colour = ref["colour"]
        n = ref["n"]

        # ±1 SD band
        ax.hlines(y, mean - sd, mean + sd, color=colour, linewidth=10, alpha=0.35, zorder=1)
        # mean marker
        ax.scatter(mean, y, s=60, color=colour, edgecolors="white", linewidths=0.8, zorder=2)

        ax.text(
            mean + 0.04, y + 0.12,
            f"{region} (n={n})",
            fontsize=9, color=colour, fontweight="bold",
            ha="left", va="bottom"
        )

    # ------------------------------------------------------------------
    # Sample points
    # ------------------------------------------------------------------
    sample_y = 0.0
    rng = np.random.default_rng(42)
    y_jit = rng.uniform(-0.18, 0.18, size=len(df))

    for i, (_, row) in enumerate(df.iterrows()):
        x = row["d13C"]
        y = sample_y + y_jit[i]

        # Flag outside broad olive envelope
        if olive_low <= x <= olive_high:
            pt_colour = "#2C3E50"
        else:
            pt_colour = "#C0392B"

        ax.scatter(
            x, y,
            s=95,
            color=pt_colour,
            edgecolors="white",
            linewidths=1.0,
            zorder=5
        )

        ax.annotate(
            str(row["SampleID"]),
            xy=(x, y),
            xytext=(x, y - 0.33),
            fontsize=10,
            ha="center",
            va="top",
            color=pt_colour,
            fontweight="bold",
            path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
            zorder=6
        )

    # Label for sample row
    ax.text(
        ax.get_xlim()[0] if ax.get_xlim() != (0.0, 1.0) else C3_RANGE[0],
        sample_y + 0.3,
        "Samples",
        fontsize=10, fontweight="bold", color="#2C3E50",
        ha="left", va="center"
    )

    # ------------------------------------------------------------------
    # Cosmetics
    # ------------------------------------------------------------------
    ax.set_yticks([])
    ax.set_xlabel(r"$\delta^{13}$C (‰ vs VPDB)", fontsize=12)
    ax.set_title(
        r"$\delta^{13}$C of Olive Oils — Geography Reference Bands with C3/C4 Screen",
        fontsize=13, fontweight="bold"
    )

    # Nice x-limits
    xmin = min(df["d13C"].min(), C3_RANGE[0], olive_low) - 0.5
    ax.set_xlim(-31, -20)

    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", linestyle=":", alpha=0.35)

    # Custom legend
    legend_handles = [
        Line2D([0], [0], color="#2E86AB", lw=8, alpha=0.35, label=f"C3 zone ({C3_RANGE[0]} to {C3_RANGE[1]}‰)"),
        Line2D([0], [0], color="#8BC34A", lw=8, alpha=0.35, label="Olive reference envelope"),
        Line2D([0], [0], color="#E76F51", lw=8, alpha=0.35, label=f"C4 zone ({C4_RANGE[0]} to {C4_RANGE[1]}‰)"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#2C3E50",
               markeredgecolor="white", markersize=9, label="Sample within olive envelope"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#C0392B",
               markeredgecolor="white", markersize=9, label="Sample outside olive envelope"),
    ]
    ax.legend(handles=legend_handles, fontsize=8.5, loc="upper right", framealpha=0.95)

    plt.tight_layout()
    out_path = OUTPUT_DIR / "irms_C13_geography_adulteration.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")